In [0]:
schema  = 'ID string ,first_name string ,last_name string ,address string ,skills string ,contacts string '

df = spark.read.format("csv")\
                .option('header' , 'true')\
                    .option('quote' , '\"')\
                        .option('escape' , '\"')\
                            .load(path = '/Volumes/dev/spark_db/datasets/spark_programming/data/students_offline.csv')


In [0]:
df.write.mode('overwrite').option("overwriteSchema" , 'true').saveAsTable('dev.spark_db.offline_students_raw')

spark.sql(''' select * from  dev.spark_db.offline_students_raw ''').display()

In [0]:
from pyspark.sql.functions import parse_json

df = df.withColumns(
    {
        'Address' : parse_json(df.Address),
        'Skills'  : parse_json(df.Skills),
        'Contacts': parse_json(df.Contacts)
    }
)
df.printSchema()

df.write.mode('overwrite').saveAsTable('dev.spark_db.offline_var_students')

In [0]:
spark.sql(

    '''
    -- variant type is case sensitive
    select id , address:Country from dev.spark_db.offline_var_students
    --   string ,  variant
    '''

).display()

In [0]:
spark.sql(

    '''
    -- Error 
    select address:Country,count(*) from dev.spark_db.offline_var_students
    group by address:Country
    '''

).display()

In [0]:
spark.sql(

    ''' 
    select cast(address:Country as string),count(*) from dev.spark_db.offline_var_students
    group by cast(address:Country as string)
    '''

).display()

In [0]:
spark.sql(

    ''' 
    --variant_explode  - table value function returns value
    select id , FirstName , LastName  ,value  , cast(value:Skill as string) , value:YearsOfExperience
    from dev.spark_db.offline_var_students , lateral variant_explode(Skills)     -- If skills columns is null for some records it removes that rows (like inner join), to avoid these kind of issue we can use variant_explode_outer(outer join)
    where cast(value:Skill as string)  like '%Spark' and cast(value:YearsOfExperience as integer) > 2 
    
    '''

).display()

In [0]:
spark.sql(

    ''' 
    select id , FirstName , LastName  ,value  , cast(value:Skill as string) , value:YearsOfExperience
    from dev.spark_db.offline_var_students , lateral variant_explode_outer(Skills) 
    where cast(value:Skill as string)  like '%Spark' and cast(value:YearsOfExperience as integer) > 2 
    
    '''

).display()

In [0]:
spark.sql(

    ''' 
    select id , FirstName , LastName , Contacts:email
    from dev.spark_db.offline_var_students
    where Contacts:whatsapp is null and Contacts:phone is null
    
    '''

).display()